In [1]:
# If these imports fail, uncomment the next line to install.
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
# !pip install transformers captum lime matplotlib sentencepiece

import sys, platform
print("Python:", sys.version)
print("OS:", platform.platform())


Python: 3.11.14 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 18:30:03) [MSC v.1929 64 bit (AMD64)]
OS: Windows-10-10.0.26100-SP0


In [2]:
import os, numpy as np, torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from captum.attr import LayerIntegratedGradients
from lime.lime_text import LimeTextExplainer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


Torch: 2.9.0+cpu
CUDA available: False


In [3]:
# === EDIT THESE TO MATCH YOUR MACHINE ===
AFRIBERTA_MODEL_DIR = r"C:\Users\Shayl\Desktop\University\Honours\Semester 2\INF 791\Assignment 3\drive-download-20251106T093018Z-1-001\trained_afriberta_model"
AFRIBERTA_TOKEN_DIR = r"C:\Users\Shayl\Desktop\University\Honours\Semester 2\INF 791\Assignment 3\drive-download-20251106T093018Z-1-001\trained_afriberta_tokenizer"

AFROXLMR_MODEL_DIR  = r"C:\Users\Shayl\Desktop\University\Honours\Semester 2\INF 791\Assignment 3\drive-download-20251106T093018Z-1-001\trained_afroxlmr_model"
AFROXLMR_TOKEN_DIR  = r"C:\Users\Shayl\Desktop\University\Honours\Semester 2\INF 791\Assignment 3\drive-download-20251106T093018Z-1-001\trained_afroxlmr_tokenizer"

OUT_DIR = r"C:\Users\Shayl\Desktop\University\Honours\Semester 2\INF 791\Assignment 3\Outputs"
os.makedirs(OUT_DIR, exist_ok=True)

for p in [AFRIBERTA_MODEL_DIR, AFRIBERTA_TOKEN_DIR, AFROXLMR_MODEL_DIR, AFROXLMR_TOKEN_DIR, OUT_DIR]:
    print(("OK  " if os.path.exists(p) else "MISS"), p)


OK   C:\Users\Shayl\Desktop\University\Honours\Semester 2\INF 791\Assignment 3\drive-download-20251106T093018Z-1-001\trained_afriberta_model
OK   C:\Users\Shayl\Desktop\University\Honours\Semester 2\INF 791\Assignment 3\drive-download-20251106T093018Z-1-001\trained_afriberta_tokenizer
OK   C:\Users\Shayl\Desktop\University\Honours\Semester 2\INF 791\Assignment 3\drive-download-20251106T093018Z-1-001\trained_afroxlmr_model
OK   C:\Users\Shayl\Desktop\University\Honours\Semester 2\INF 791\Assignment 3\drive-download-20251106T093018Z-1-001\trained_afroxlmr_tokenizer
OK   C:\Users\Shayl\Desktop\University\Honours\Semester 2\INF 791\Assignment 3\Outputs


In [4]:
def load_model_and_tokenizer(model_dir, tok_dir):
    tok = AutoTokenizer.from_pretrained(tok_dir, use_fast=False)
    mdl = AutoModelForSequenceClassification.from_pretrained(model_dir).to(device).eval()
    return tok, mdl

tok_afriberta, mdl_afriberta = load_model_and_tokenizer(AFRIBERTA_MODEL_DIR, AFRIBERTA_TOKEN_DIR)
tok_afroxlmr,  mdl_afroxlmr  = load_model_and_tokenizer(AFROXLMR_MODEL_DIR,  AFROXLMR_TOKEN_DIR)

# label mapping (fallback if not present)
id2label = getattr(mdl_afriberta.config, "id2label", {0:"negative", 1:"neutral", 2:"positive"})
label2id = {v:k for k,v in id2label.items()}
print("Labels:", id2label)


Labels: {0: 'negative', 1: 'neutral', 2: 'positive'}


In [5]:
samples = [
    ("zu", "Lena ifilimu imnandi kakhulu."),              # Zulu (positive)
    ("xh", "Le mveliso imbi kakhulu, andiyithandi."),     # Xhosa (negative)
    ("sn", "Chikafu ichi chakashata, handisi kufara."),    # Shona (negative)
    ("st", "Tshebeletso e ntle haholo, ke khotsofetse."), # Sesotho (positive)
    ("af", "Die diens was uitstekend en vinnig."),         # Afrikaans (positive)
    ("en", "Not great at all—totally disappointing."),     # English (negative with negation)
]
len(samples)


6

In [6]:
def predict(model, tokenizer, text):
    enc = tokenizer(text, return_tensors='pt', truncation=True, max_length=256).to(device)
    with torch.no_grad():
        out = model(**enc)
        probs = F.softmax(out.logits, dim=-1)[0].detach().cpu().numpy()
    pred_id = int(np.argmax(probs))
    return pred_id, probs, enc


In [7]:
def plot_attention_heatmap(model, tokenizer, enc, title, save_path):
    # get last-layer attentions (mean over heads)
    with torch.no_grad():
        outputs = model.base_model(**enc, output_attentions=True)

    attn_last = outputs.attentions[-1]        # [B, H, S, S]
    attn_mean = attn_last.mean(dim=1)[0].detach().cpu().numpy()

    tokens = tokenizer.convert_ids_to_tokens(enc['input_ids'][0].cpu())
    fig_w = min(16, 0.6*len(tokens)); fig_h = min(12, 0.6*len(tokens))

    fig, ax = plt.subplots(figsize=(max(6, fig_w), max(5, fig_h)))
    im = ax.imshow(attn_mean, aspect='auto')
    ax.set_xticks(range(len(tokens))); ax.set_xticklabels(tokens, rotation=90)
    ax.set_yticks(range(len(tokens))); ax.set_yticklabels(tokens)
    ax.set_title(title)
    fig.colorbar(im)
    plt.tight_layout(); plt.savefig(save_path, dpi=220); plt.close(fig)


In [8]:
def ig_token_importance(model, tokenizer, text, target=None):
    class Wrapped(torch.nn.Module):
        def __init__(self, mdl): 
            super().__init__(); self.m=mdl
        def forward(self, input_ids, attention_mask):
            return self.m(input_ids=input_ids, attention_mask=attention_mask).logits

    wrapped = Wrapped(model).eval()

    enc = tokenizer(text, return_tensors='pt', truncation=True, max_length=256)
    for k in enc: enc[k] = enc[k].to(device)

    pred = model(**enc).logits.argmax(dim=-1).item()
    target = pred if target is None else target

    lig = LayerIntegratedGradients(wrapped, wrapped.m.base_model.embeddings)
    attributions, delta = lig.attribute(
        inputs=(enc['input_ids'], enc['attention_mask']),
        target=target, n_steps=24, return_convergence_delta=True
    )

    token_attr = attributions[0].sum(dim=-1).squeeze().detach().cpu().numpy()
    tokens = tokenizer.convert_ids_to_tokens(enc['input_ids'][0].detach().cpu())
    token_attr = token_attr / (np.abs(token_attr).max() + 1e-8)
    return tokens, token_attr, pred

def plot_token_bars(tokens, scores, title, save_path, top_k=25):
    pairs = [(t, s) for t, s in zip(tokens, scores)
             if not (t.startswith("<") or t in {"[CLS]","[SEP]","</s>","<s>","[PAD]"} )]
    pairs = sorted(pairs, key=lambda x: abs(x[1]), reverse=True)[:top_k]
    if not pairs: 
        print("No tokens to plot."); 
        return
    tok, sc = zip(*pairs)
    plt.figure(figsize=(10, max(3, 0.35*len(tok))))
    plt.barh(range(len(tok)), sc)
    plt.yticks(range(len(tok)), tok)
    plt.xlabel("Integrated Gradients attribution")
    plt.title(title)
    plt.tight_layout(); plt.savefig(save_path, dpi=220); plt.close()


In [9]:
def lime_explain(model, tokenizer, text, class_names):
    clf = pipeline("text-classification",
                   model=model, tokenizer=tokenizer,
                   device=0 if torch.cuda.is_available() else -1,
                   truncation=True)
    explainer = LimeTextExplainer(class_names=class_names)
    exp = explainer.explain_instance(text, clf, num_features=10)
    return exp  # exp.as_list() → [(token, weight), ...]


In [10]:
lang, text = samples[0]
print(f"Example → [{lang}] {text}")

pred_id, probs, enc = predict(mdl_afriberta, tok_afriberta, text)
print("AfriBERTa pred:", id2label.get(pred_id, pred_id), "| probs:", dict(zip([id2label.get(i,str(i)) for i in range(len(probs))], probs.round(3))))

attn_png = os.path.join(OUT_DIR, f"AfriBERTa_{lang}_ATTN.png")
plot_attention_heatmap(mdl_afriberta, tok_afriberta, enc,
                       title=f"AfriBERTa attention: {lang}",
                       save_path=attn_png)
print("Saved:", attn_png)

tokens, ig_scores, _ = ig_token_importance(mdl_afriberta, tok_afriberta, text)
ig_png = os.path.join(OUT_DIR, f"AfriBERTa_{lang}_IG.png")
plot_token_bars(tokens, ig_scores,
                title=f"AfriBERTa IG token importance: {lang}",
                save_path=ig_png)
print("Saved:", ig_png)

# LIME (optional)
try:
    class_names = [id2label.get(i,str(i)) for i in range(len(probs))]
    exp = lime_explain(mdl_afriberta, tok_afriberta, text, class_names)
    toks, wts = zip(*exp.as_list()) if exp.as_list() else ([],[])
    plt.figure(figsize=(8, max(3, 0.4*len(toks))))
    plt.barh(range(len(toks)), wts); plt.yticks(range(len(toks)), toks)
    plt.title(f"AfriBERTa LIME: {lang}")
    lime_png = os.path.join(OUT_DIR, f"AfriBERTa_{lang}_LIME.png")
    plt.tight_layout(); plt.savefig(lime_png, dpi=220); plt.close()
    print("Saved:", lime_png)
except Exception as e:
    print("LIME failed:", e)


Example → [zu] Lena ifilimu imnandi kakhulu.


XLMRobertaSdpaSelfAttention is used but `torch.nn.functional.scaled_dot_product_attention` does not support non-absolute `position_embedding_type` or `output_attentions=True` or `head_mask`. Falling back to the manual attention implementation, but specifying the manual implementation will be required from Transformers version v5.0.0 onwards. This warning can be removed using the argument `attn_implementation="eager"` when loading the model.


AfriBERTa pred: negative | probs: {'negative': 0.575, 'neutral': 0.277, 'positive': 0.148}
Saved: C:\Users\Shayl\Desktop\University\Honours\Semester 2\INF 791\Assignment 3\Outputs\AfriBERTa_zu_ATTN.png


Device set to use cpu


Saved: C:\Users\Shayl\Desktop\University\Honours\Semester 2\INF 791\Assignment 3\Outputs\AfriBERTa_zu_IG.png


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


LIME failed: list indices must be integers or slices, not tuple


In [11]:
USE_LIME = False  # <- leave False

import csv, os

def run_all(model, tokenizer, model_tag, samples):
    rows = []
    for lang, text in samples:
        pred_id, probs, enc = predict(model, tokenizer, text)
        pred_label = id2label.get(pred_id, str(pred_id))
        pstr = ", ".join([f"{id2label.get(i,str(i))}:{probs[i]:.3f}" for i in range(len(probs))])

        # Attention
        attn_path = os.path.join(OUT_DIR, f"{model_tag}_{lang}_ATTN.png")
        plot_attention_heatmap(model, tokenizer, enc,
                               title=f"{model_tag} attention: {lang} | '{text[:60]}...'",
                               save_path=attn_path)

        # Integrated Gradients
        tokens, ig_scores, _ = ig_token_importance(model, tokenizer, text)
        ig_path = os.path.join(OUT_DIR, f"{model_tag}_{lang}_IG.png")
        plot_token_bars(tokens, ig_scores,
                        title=f"{model_tag} IG token importance: {lang}",
                        save_path=ig_path)

        lime_path = "LIME_skipped"
        rows.append([model_tag, lang, text, pred_label, pstr, attn_path, ig_path, lime_path])

    csv_path = os.path.join(OUT_DIR, f"{model_tag}_summary.csv")
    with open(csv_path, "w", newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(["model","lang","text","pred_label","probs","attention_png","ig_png","lime_png"])
        writer.writerows(rows)
    print("Saved summary:", csv_path)

# Run both models
run_all(mdl_afriberta, tok_afriberta, "AfriBERTa", samples)
run_all(mdl_afroxlmr,  tok_afroxlmr,  "AfroXLMR",  samples)
print("All images & CSVs saved to:", OUT_DIR)


Saved summary: C:\Users\Shayl\Desktop\University\Honours\Semester 2\INF 791\Assignment 3\Outputs\AfriBERTa_summary.csv
Saved summary: C:\Users\Shayl\Desktop\University\Honours\Semester 2\INF 791\Assignment 3\Outputs\AfroXLMR_summary.csv
All images & CSVs saved to: C:\Users\Shayl\Desktop\University\Honours\Semester 2\INF 791\Assignment 3\Outputs
